In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import make_scorer, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier, BaggingClassifier
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix
from sklearn.metrics import matthews_corrcoef
from sklearn.metrics import cohen_kappa_score
from lightgbm import LGBMClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings("ignore")

In [ ]:

from google.colab import files
uploaded = files.upload()

Saving bank-additional-full.csv to bank-additional-full.csv


In [ ]:
#File Upload
df=pd.read_csv("bank-additional-full.csv", sep=';')

In [ ]:
# 3. Drop Leakage Feature
df.drop('duration', axis=1, inplace=True)

In [ ]:
# 4. Handle "unknown" Values
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].replace('unknown', df[col].mode()[0])

In [ ]:
# 5. Encode Target
df['y'] = df['y'].map({'no': 0, 'yes': 1})
# 6. One-Hot Encoding
df = pd.get_dummies(df, drop_first=True)
X_shuffled = df.drop('y', axis=1)
y_shuffled = df['y']

In [ ]:
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from scipy.stats import uniform

# Split
X_train, X_test, y_train, y_test = train_test_split(X_shuffled, y_shuffled, test_size=0.2, random_state=42)

In [ ]:
# Model
from sklearn.ensemble import RandomForestClassifier
model1 = RandomForestClassifier()

# Search space
param_dist2 = {
         "n_estimators":[100,200,300],
    "max_depth":[10,20,None],
    "min_samples_split":[2,5],
    "min_samples_leaf":[1,2]
}

In [ ]:
# RandomForestClassifier

random_search = RandomizedSearchCV(model1, param_distributions=param_dist2, n_iter=15,
                                   cv=10, scoring="accuracy", random_state=42, n_jobs=-1)
random_search.fit(X_train, y_train)

best_model_LR = random_search.best_estimator_
print("Best parameters:", random_search.best_params_)
print("Best CV accuracy: {:.4f}".format(random_search.best_score_))
print("Test accuracy: {:.4f}".format(best_model_LR.score(X_test, y_test)))

Best parameters: {'n_estimators': 100, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_depth': 10}
Best CV accuracy: 0.9017
Test accuracy: 0.8967


In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, matthews_corrcoef,
    cohen_kappa_score, balanced_accuracy_score
)
import numpy as np
import pandas as pd
import time

# Define your classifier here (Random Forest)
from sklearn.ensemble import RandomForestClassifier
clf_name = "Random Forest"
clf = RandomForestClassifier()



# Store results
results1 = []

# KFold Cross-Validation
from sklearn.model_selection import KFold
kf = KFold(n_splits=10, shuffle=True, random_state=42)

for fold, (train_idx, test_idx) in enumerate(kf.split(X_shuffled)):
    X_train, X_test = X_shuffled.iloc[train_idx], X_shuffled.iloc[test_idx]
    y_train, y_test = y_shuffled.iloc[train_idx], y_shuffled.iloc[test_idx]

    # Train
    start_time = time.time()
    clf.fit(X_train, y_train)
    training_time = time.time() - start_time

    # Predict
    y_pred = clf.predict(X_test)

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) != 0 else 0
    gm = np.sqrt(rec * specificity)
    auc = roc_auc_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)
    kappa = cohen_kappa_score(y_test, y_pred)
    balanced_acc = balanced_accuracy_score(y_test, y_pred)

    # Store fold results
    results1.append({
        "Fold": fold+1,
        "Classifier": clf_name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "Specificity": specificity,
        "F1": f1,
        "GM": gm,
        "FPR": fpr,
        "AUC": auc,
        "MCC": mcc,
        "Kappa": kappa,
        "Balanced Accuracy": balanced_acc,
        "Training Time (s)": training_time
    })

# Create DataFrame
results1_df1 = pd.DataFrame(results1)

# Print
print(f"Results for {clf_name}:")
print(results1_df1)

# Save to CSV if needed
results1_df1.to_csv(f"{clf_name.replace(' ', '_')}_metrics.csv", index=False)

Results for Random Forest:
   Fold     Classifier  Accuracy  Precision    Recall  Specificity        F1  \
0     1  Random Forest  0.891964   0.532075  0.305195     0.966092  0.387895   
1     2  Random Forest  0.889051   0.533333  0.270613     0.969281  0.359046   
2     3  Random Forest  0.898276   0.576419  0.290749     0.973533  0.386530   
3     4  Random Forest  0.889051   0.513109  0.295259     0.964432  0.374829   
4     5  Random Forest  0.887837   0.490842  0.293217     0.962043  0.367123   
5     6  Random Forest  0.885166   0.556777  0.301587     0.966528  0.391248   
6     7  Random Forest  0.897548   0.518349  0.262791     0.971537  0.348765   
7     8  Random Forest  0.899005   0.562992  0.319196     0.969763  0.407407   
8     9  Random Forest  0.888538   0.511013  0.250000     0.969622  0.335745   
9    10  Random Forest  0.890481   0.559567  0.320248     0.966428  0.407359   

         GM       FPR       AUC       MCC     Kappa  Balanced Accuracy  \
0  0.542998  0.033

In [ ]:

from sklearn.svm import LinearSVC

model4 = LinearSVC()

param_dist4 = {
    "C": [0.01, 0.1, 1, 10],
    "loss": ["hinge", "squared_hinge"],
    "max_iter": [1000, 2000]
}

In [ ]:
# Model
from sklearn.svm import LinearSVC
model2 = LinearSVC()

from sklearn.svm import LinearSVC

model4 = LinearSVC()

param_dist4 = {
    "C": [0.01, 0.1, 1, 10],
    "loss": ["hinge", "squared_hinge"],
    "max_iter": [1000, 2000]
}

In [ ]:
# LinearSVC

random_search = RandomizedSearchCV(model2, param_distributions=param_dist4, n_iter=10,
                                   cv=10, scoring="accuracy", random_state=42, n_jobs=-1)
random_search.fit(X_train, y_train)

best_model_LR = random_search.best_estimator_
print("Best parameters:", random_search.best_params_)
print("Best CV accuracy: {:.4f}".format(random_search.best_score_))
print("Test accuracy: {:.4f}".format(best_model_LR.score(X_test, y_test)))

Best parameters: {'max_iter': 2000, 'loss': 'hinge', 'C': 0.01}
Best CV accuracy: 0.8757
Test accuracy: 0.8970


In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, matthews_corrcoef,
    cohen_kappa_score, balanced_accuracy_score
)
import numpy as np
import pandas as pd
import time

# Define your classifier here (LinearSVC)
from sklearn.svm import LinearSVC
clf_name = "SVM"
clf = LinearSVC()


# Store results
results2 = []



# KFold Cross-Validation
from sklearn.model_selection import KFold
kf = KFold(n_splits=15, shuffle=True, random_state=42)

for fold, (train_idx, test_idx) in enumerate(kf.split(X_shuffled)):
    X_train, X_test = X_shuffled.iloc[train_idx], X_shuffled.iloc[test_idx]
    y_train, y_test = y_shuffled.iloc[train_idx], y_shuffled.iloc[test_idx]

    # Train
    start_time = time.time()
    clf.fit(X_train, y_train)
    training_time = time.time() - start_time

    # Predict
    y_pred = clf.predict(X_test)

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) != 0 else 0
    gm = np.sqrt(rec * specificity)
    auc = roc_auc_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)
    kappa = cohen_kappa_score(y_test, y_pred)
    balanced_acc = balanced_accuracy_score(y_test, y_pred)

    # Store fold results
    results2.append({
        "Fold": fold+1,
        "Classifier": clf_name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "Specificity": specificity,
        "F1": f1,
        "GM": gm,
        "FPR": fpr,
        "AUC": auc,
        "MCC": mcc,
        "Kappa": kappa,
        "Balanced Accuracy": balanced_acc,
        "Training Time (s)": training_time
    })
# Create DataFrame
results2_df2 = pd.DataFrame(results2)

# Print
print(f"Results for {clf_name}:")
print(results2_df2)

# Save to CSV if needed
results2_df2.to_csv(f"{clf_name.replace(' ', '_')}_metrics.csv", index=False)

Results for SVM:
    Fold Classifier  Accuracy  Precision    Recall  Specificity        F1  \
0      1        SVM  0.897669   0.714286  0.203125     0.989283  0.316302   
1      2        SVM  0.895849   0.647727  0.182692     0.987264  0.285000   
2      3        SVM  0.897669   0.627907  0.178218     0.986901  0.277635   
3      4        SVM  0.904588   0.747368  0.229773     0.990152  0.351485   
4      5        SVM  0.899490   0.691358  0.182410     0.989750  0.288660   
5      6        SVM  0.900583   0.646465  0.211921     0.985679  0.319202   
6      7        SVM  0.901311   0.602151  0.193103     0.984935  0.292428   
7      8        SVM  0.891843   0.714286  0.193452     0.989212  0.304450   
8      9        SVM  0.894028   0.720000  0.214925     0.988387  0.331034   
9     10        SVM  0.900947   0.670213  0.207237     0.987305  0.316583   
10    11        SVM  0.908230   0.723077  0.167260     0.992698  0.271676   
11    12        SVM  0.907502   0.729412  0.211604     0.99

In [ ]:
# Model

from xgboost import XGBClassifier

model3 =  XGBClassifier(use_label_encoder=False, eval_metric='logloss')

param_dist5 = {
    "n_estimators": [100, 200],
    "learning_rate": [0.01, 0.1],
            "max_depth": [3, 6]
}

In [ ]:
# XGBoost

random_search = RandomizedSearchCV(model3, param_distributions=param_dist5, n_iter=100,
                                   cv=20, scoring="accuracy", random_state=42, n_jobs=-1)
random_search.fit(X_train, y_train)

best_model_LR = random_search.best_estimator_
print("Best parameters:", random_search.best_params_)
print("Best CV accuracy: {:.4f}".format(random_search.best_score_))
print("Test accuracy: {:.4f}".format(best_model_LR.score(X_test, y_test)))

Best parameters: {'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.01}
Best CV accuracy: 0.8065
Test accuracy: 0.8944


In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, matthews_corrcoef,
    cohen_kappa_score, balanced_accuracy_score
)
import numpy as np
import pandas as pd
import time

# Define your classifier here (XGBoost)
from xgboost import XGBClassifier
clf_name = "XGBoost"
clf =  XGBClassifier(use_label_encoder=False, eval_metric='logloss')

# Store results
results3 = []

# KFold Cross-Validation
from sklearn.model_selection import KFold
kf = KFold(n_splits=15, shuffle=True, random_state=42)

for fold, (train_idx, test_idx) in enumerate(kf.split(X_shuffled)):
    X_train, X_test = X_shuffled.iloc[train_idx], X_shuffled.iloc[test_idx]
    y_train, y_test = y_shuffled.iloc[train_idx], y_shuffled.iloc[test_idx]

    # Train
    start_time = time.time()
    clf.fit(X_train, y_train)
    training_time = time.time() - start_time

    # Predict
    y_pred = clf.predict(X_test)

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) != 0 else 0
    gm = np.sqrt(rec * specificity)
    auc = roc_auc_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)
    kappa = cohen_kappa_score(y_test, y_pred)
    balanced_acc = balanced_accuracy_score(y_test, y_pred)

    # Store fold results
    results3.append({
        "Fold": fold+1,
        "Classifier": clf_name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "Specificity": specificity,
        "F1": f1,
        "GM": gm,
        "FPR": fpr,
        "AUC": auc,
        "MCC": mcc,
        "Kappa": kappa,
        "Balanced Accuracy": balanced_acc,
        "Training Time (s)": training_time
    })

# Create DataFrame
results3_df3 = pd.DataFrame(results3)

# Print
print(f"Results for {clf_name}:")
print(results3_df3)

# Save to CSV if needed
results3_df3.to_csv(f"{clf_name.replace(' ', '_')}_metrics.csv", index=False)

Results for XGBoost:
    Fold Classifier  Accuracy  Precision    Recall  Specificity        F1  \
0      1    XGBoost  0.890386   0.559748  0.278125     0.971146  0.371608   
1      2    XGBoost  0.891843   0.551020  0.259615     0.972884  0.352941   
2      3    XGBoost  0.896941   0.571429  0.264026     0.975440  0.361174   
3      4    XGBoost  0.904953   0.671429  0.304207     0.981124  0.418708   
4      5    XGBoost  0.901311   0.625000  0.293160     0.977860  0.399113   
5      6    XGBoost  0.898034   0.568750  0.301325     0.971768  0.393939   
6      7    XGBoost  0.895484   0.509804  0.268966     0.969463  0.352144   
7      8    XGBoost  0.889658   0.610738  0.270833     0.975934  0.375258   
8      9    XGBoost  0.886016   0.572368  0.259701     0.973040  0.357290   
9     10    XGBoost  0.898034   0.593750  0.250000     0.978706  0.351852   
10    11    XGBoost  0.906409   0.592308  0.274021     0.978499  0.374696   
11    12    XGBoost  0.907138   0.631944  0.310580     

In [ ]:
# Model LightGBM

model4 =  LGBMClassifier()

param_dist6 = {
    "n_estimators": [100, 200],
    "learning_rate": [0.01, 0.1]
}

In [ ]:
# LightGBM

random_search = RandomizedSearchCV(model4, param_distributions=param_dist6, n_iter=100,
                                   cv=25, scoring="accuracy", random_state=42, n_jobs=-1)
random_search.fit(X_train, y_train)

best_model_LR = random_search.best_estimator_
print("Best parameters:", random_search.best_params_)
print("Best CV accuracy: {:.4f}".format(random_search.best_score_))
print("Test accuracy: {:.4f}".format(best_model_LR.score(X_test, y_test)))

[LightGBM] [Info] Number of positive: 4327, number of negative: 34116
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005079 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 492
[LightGBM] [Info] Number of data points in the train set: 38443, number of used features: 44
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.112556 -> initscore=-2.064892
[LightGBM] [Info] Start training from score -2.064892
Best parameters: {'n_estimators': 100, 'learning_rate': 0.01}
Best CV accuracy: 0.6616
Test accuracy: 0.8947


In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, matthews_corrcoef,
    cohen_kappa_score, balanced_accuracy_score
)
import numpy as np
import pandas as pd
import time

# Define your classifier here (XGBoost)
from lightgbm import LGBMClassifier
clf_name = "LightGBM"
clf =  LGBMClassifier()


# Store results
results4 = []



# KFold Cross-Validation
from sklearn.model_selection import KFold
kf = KFold(n_splits=15, shuffle=True, random_state=42)

for fold, (train_idx, test_idx) in enumerate(kf.split(X_shuffled)):
    X_train, X_test = X_shuffled.iloc[train_idx], X_shuffled.iloc[test_idx]
    y_train, y_test = y_shuffled.iloc[train_idx], y_shuffled.iloc[test_idx]

    # Train
    start_time = time.time()
    clf.fit(X_train, y_train)
    training_time = time.time() - start_time

    # Predict
    y_pred = clf.predict(X_test)

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) != 0 else 0
    gm = np.sqrt(rec * specificity)
    auc = roc_auc_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)
    kappa = cohen_kappa_score(y_test, y_pred)
    balanced_acc = balanced_accuracy_score(y_test, y_pred)

    # Store fold results
    results4.append({
        "Fold": fold+1,
        "Classifier": clf_name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "Specificity": specificity,
        "F1": f1,
        "GM": gm,
        "FPR": fpr,
        "AUC": auc,
        "MCC": mcc,
        "Kappa": kappa,
        "Balanced Accuracy": balanced_acc,
        "Training Time (s)": training_time
    })

# Create DataFrame
results4_df4 = pd.DataFrame(results4)

# Print
print(f"Results for {clf_name}:")
print(results4_df4)

# Save to CSV if needed
results4_df4.to_csv(f"{clf_name.replace(' ', '_')}_metrics.csv", index=False)

[LightGBM] [Info] Number of positive: 4320, number of negative: 34122
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004853 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 493
[LightGBM] [Info] Number of data points in the train set: 38442, number of used features: 44
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.112377 -> initscore=-2.066687
[LightGBM] [Info] Start training from score -2.066687
[LightGBM] [Info] Number of positive: 4328, number of negative: 34114
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005487 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 493
[LightGBM] [Info] Number of data points in the train set: 38442, number of used features: 44
[LightGBM] [Info] [bin

In [ ]:
from scipy.stats import wilcoxon
import pandas as pd
import itertools

print("\nWILCOXON SIGNED-RANK TEST")

#  Create ONE master dataframe (paired folds)
acc_df = pd.DataFrame({
    "Random Forest": results1_df1["Accuracy"].reset_index(drop=True),
    "SVM": results2_df2["Accuracy"].reset_index(drop=True),
    "XGBoost": results3_df3["Accuracy"].reset_index(drop=True),
    "LightGBM": results4_df4["Accuracy"].reset_index(drop=True),
})

#  Remove rows with missing folds (VERY IMPORTANT)
acc_df = acc_df.dropna()
print("Paired dataset shape:", acc_df.shape)

#  Pairwise Wilcoxon test
wilcoxon_results = []

for m1, m2 in itertools.combinations(acc_df.columns, 2):

    arr1 = acc_df[m1].values
    arr2 = acc_df[m2].values

    stat, p = wilcoxon(arr1, arr2)

    wilcoxon_results.append([m1, m2, stat, p])

#  Result table
wilcoxon_df = pd.DataFrame(
    wilcoxon_results,
    columns=["Model 1", "Model 2", "Statistic", "p-value"]
)

wilcoxon_df["Significant (p<0.05)"] = wilcoxon_df["p-value"] < 0.05

print(wilcoxon_df)

#  Save for research paper
wilcoxon_df.to_excel("Wilcoxon_Test4.xlsx", index=False)

print("Wilcoxon test completed successfully ")


WILCOXON SIGNED-RANK TEST
Paired dataset shape: (10, 4)
         Model 1   Model 2  Statistic   p-value  Significant (p<0.05)
0  Random Forest       SVM        7.0  0.037109                  True
1  Random Forest   XGBoost       17.0  0.322266                 False
2  Random Forest  LightGBM        1.0  0.003906                  True
3            SVM   XGBoost        4.0  0.013672                  True
4            SVM  LightGBM       16.0  0.271484                 False
5        XGBoost  LightGBM        0.0  0.001953                  True
Wilcoxon test completed successfully 


In [ ]:
summary = pd.DataFrame({
    "Models":[
        "Random Forest","SVM","XGBoost","LightGBM"],
    "Accuracy":[
        results1_df1["Accuracy"].mean(),
        results2_df2["Accuracy"].mean(),
        results3_df3["Accuracy"].mean(),
        results4_df4["Accuracy"].mean()
    ],
    "Precision":[
        results1_df1["Precision"].mean(),
        results2_df2["Precision"].mean(),
        results3_df3["Precision"].mean(),
        results4_df4["Precision"].mean()
    ],
    "Recall":[
        results1_df1["Recall"].mean(),
        results2_df2["Recall"].mean(),
        results3_df3["Recall"].mean(),
        results4_df4["Recall"].mean()
    ],
    "F1":[
        results1_df1["F1"].mean(),
        results2_df2["F1"].mean(),
        results3_df3["F1"].mean(),
        results4_df4["F1"].mean()
    ],
    "AUC":[
        results1_df1["AUC"].mean(),
        results2_df2["AUC"].mean(),
        results3_df3["AUC"].mean(),
        results4_df4["AUC"].mean()
    ],
    "Training Time":[
        results1_df1["Training Time (s)"].mean(),
        results2_df2["Training Time (s)"].mean(),
        results3_df3["Training Time (s)"].mean(),
        results4_df4["Training Time (s)"].mean()
    ]
})

print(summary)

          Models  Accuracy  Precision    Recall        F1       AUC  \
0  Random Forest  0.891692   0.535448  0.290885  0.376595  0.629406   
1            SVM  0.899631   0.695758  0.194623  0.303481  0.591867   
2        XGBoost  0.897446   0.595768  0.281162  0.381647  0.628432   
3       LightGBM  0.900238   0.634629  0.269816  0.378255  0.625045   

   Training Time  
0       4.331812  
1       0.213695  
2       0.702840  
3       0.553440  


In [ ]:
import numpy as np

print("\nTOPSIS RANKING")

# remove model column
data = summary.drop("Models", axis=1).values

# Step 1: Normalize matrix
norm = data / np.sqrt((data**2).sum(axis=0))

# Step 2: Weights (equal weights)
weights = np.ones(norm.shape[1]) / norm.shape[1]
weighted = norm * weights

# Step 3: Ideal best & worst
# Benefit metrics: Accuracy, Precision, Recall, F1, AUC → MAX
# Cost metric: Training Time → MIN

ideal_best = np.max(weighted, axis=0)
ideal_worst = np.min(weighted, axis=0)

# For Training Time column (last column) reverse
ideal_best[-1] = np.min(weighted[:, -1])
ideal_worst[-1] = np.max(weighted[:, -1])

# Step 4: Distance to ideal solutions
dist_best = np.sqrt(((weighted - ideal_best)**2).sum(axis=1))
dist_worst = np.sqrt(((weighted - ideal_worst)**2).sum(axis=1))

# Step 5: TOPSIS score
topsis_score = dist_worst / (dist_best + dist_worst)

summary["TOPSIS Score"] = topsis_score
summary["Rank"] = summary["TOPSIS Score"].rank(ascending=False)

summary = summary.sort_values("Rank")

print(summary)

summary.to_excel("TOPSIS_Ranking4.xlsx", index=False)


TOPSIS RANKING
          Models  Accuracy  Precision    Recall        F1       AUC  \
3       LightGBM  0.900238   0.634629  0.269816  0.378255  0.625045   
2        XGBoost  0.897446   0.595768  0.281162  0.381647  0.628432   
1            SVM  0.899631   0.695758  0.194623  0.303481  0.591867   
0  Random Forest  0.891692   0.535448  0.290885  0.376595  0.629406   

   Training Time  TOPSIS Score  Rank  
3       0.553440      0.897564   1.0  
2       0.702840      0.859450   2.0  
1       0.213695      0.813418   3.0  
0       4.331812      0.184158   4.0  


In [ ]:
models_results4 = {
    "Random Forest": results1_df1,
    "SVM": results2_df2,
    "XGBoost": results3_df3,
    "LightGBM": results4_df4
}

In [ ]:
metrics = [
    "Accuracy","Precision","Recall","Specificity",
    "F1","GM","FPR","AUC","MCC","Kappa",
    "Balanced Accuracy","Training Time (s)"
]

In [ ]:
print("Creating FINAL Excel file...")

with pd.ExcelWriter("Final_Results_best4.xlsx") as writer:

    # Loop through each metric → each becomes a sheet
    for metric in metrics:

        metric_df = pd.DataFrame()

        # Collect metric from every model
        for model_name, df in models_results4.items():
            metric_df[model_name] = df[metric].reset_index(drop=True)

        # Add fold column (1–10)
        metric_df.insert(0, "Fold", range(1, len(metric_df)+1))

        # Save sheet
        metric_df.to_excel(writer, sheet_name=metric, index=False)

print("Excel file created successfully ")

Creating FINAL Excel file...
Excel file created successfully 


In [ ]:
from google.colab import files
files.download("Final_Results_best4.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files
files.download("TOPSIS_Ranking4.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files
files.download("Wilcoxon_Test4.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>